# Quantifying Home-Field Advantage in College Football

This notebook builds three machine learning models to quantify home-field advantage:

1. **Logistic Regression**: Predicts the probability of a home team win
2. **Ridge Regression**: Predicts the expected point margin (L2 regularization)
3. **LASSO Regression**: Predicts the expected point margin (L1 regularization with feature selection)

We will use these models to estimate how much playing at home improves a team's chances of winning and their expected score margin in an evenly-matched game. We'll compare Ridge vs. LASSO to determine which provides better predictions and interpretability.

## 1. Imports and Configuration

Import all required libraries and set random seeds for reproducibility.

In [49]:
# Standard libraries
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path

# Scikit-learn imports
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)

# Model persistence
import joblib

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load Raw Data

Load the merged game data from the cleaned dataset.

In [30]:
# Load the merged games dataset
data_path = '../data/cleaned/merged_games.csv'
merged = pd.read_csv(data_path, low_memory=False)

print(f"Dataset shape: {merged.shape}")
print(f"Columns: {len(merged.columns)}")
print("\nFirst few rows:")
merged.head()

Dataset shape: (16547, 130)
Columns: 130

First few rows:


,Unnamed: 0,sched_game_id,season,week,date_key,time_key_sched,home_sched,away_sched,attendance,cfbd_game_id,...,countryCode,timezone,latitude,longitude,elevation,constructionYear,game_dt_utc,game_dt_et_cfbd_full,date_key_cfbd_full,time_key_cfbd_full
0,0,0,2002,1.0,2002-08-22,2002-08-22 19:30:00,Virginia,Colorado State,57120.0,0,...,US,America/New_York,38.031180,-78.513790,170.467743,1931.0,2002-08-22 23:30:00+00:00,2002-08-22 19:30:00,2002-08-22,2002-08-22 19:30:00
1,1,13372,2018,7.0,2018-10-13,2018-10-13 12:00:00,Auburn,Tennessee,84589.0,24485,...,US,America/Chicago,32.602553,-85.489748,201.173798,1939.0,2018-10-13 16:00:00+00:00,2018-10-13 12:00:00,2018-10-13,2018-10-13 12:00:00
2,2,13341,2018,6.0,2018-10-06,2018-10-06 16:00:00,Ohio State,Indiana,104193.0,24428,...,US,America/New_York,40.001645,-83.019727,216.677032,1922.0,2018-10-06 20:00:00+00:00,2018-10-06 16:00:00,2018-10-06,2018-10-06 16:00:00
3,3,13342,2018,6.0,2018-10-06,2018-10-06 16:00:00,UNLV,New Mexico,18949.0,24420,...,US,America/Los_Angeles,36.085935,-115.017310,490.062225,1971.0,2018-10-06 20:00:00+00:00,2018-10-06 16:00:00,2018-10-06,2018-10-06 16:00:00
4,4,13344,2018,6.0,2018-10-06,2018-10-06 16:00:00,Colorado,Arizona State,52681.0,24419,...,US,America/Denver,40.009475,-105.266905,1634.041138,1924.0,2018-10-06 20:00:00+00:00,2018-10-06 16:00:00,2018-10-06,2018-10-06 16:00:00


## 3. Data Cleaning and Target Creation

Filter the data to include only completed, non-neutral-site, regular-season games, and create our target variables:
- `home_win`: Binary indicator (1 if home team won, 0 otherwise)
- `home_margin`: Point differential (home score - away score)

In [31]:
def prepare_model_data(merged):
    """
    Clean and prepare the data for modeling.
    
    Filters to completed, non-neutral-site, regular-season games.
    Creates target variables: home_win and home_margin.
    
    Parameters:
    -----------
    merged : pd.DataFrame
        Raw merged games data
        
    Returns:
    --------
    pd.DataFrame
        Cleaned data with targets
    """
    # Make a copy to avoid modifying the original
    df = merged.copy()
    
    print(f"Starting with {len(df)} games")
    
    # Filter to completed games
    if 'completed' in df.columns:
        df = df[df['completed'] == True].copy()
        print(f"After filtering to completed games: {len(df)}")
    
    # Filter to regular season games
    if 'seasonType' in df.columns:
        df = df[df['seasonType'] == 'regular'].copy()
        print(f"After filtering to regular season: {len(df)}")
    
    # Filter out neutral site games
    if 'neutralSite' in df.columns:
        df = df[df['neutralSite'] == False].copy()
        print(f"After removing neutral site games: {len(df)}")
    elif 'neutral' in df.columns:
        df = df[df['neutral'] == False].copy()
        print(f"After removing neutral site games: {len(df)}")
    
    # Identify score columns (could be homePoints/awayPoints or score_home/score_away)
    home_score_col = None
    away_score_col = None
    
    if 'homePoints' in df.columns and 'awayPoints' in df.columns:
        home_score_col = 'homePoints'
        away_score_col = 'awayPoints'
    elif 'score_home' in df.columns and 'score_away' in df.columns:
        home_score_col = 'score_home'
        away_score_col = 'score_away'
    else:
        raise ValueError("Could not find home/away score columns")
    
    # Drop rows with missing scores
    df = df.dropna(subset=[home_score_col, away_score_col]).copy()
    print(f"After removing games with missing scores: {len(df)}")
    
    # Ensure scores are numeric
    df[home_score_col] = pd.to_numeric(df[home_score_col], errors='coerce')
    df[away_score_col] = pd.to_numeric(df[away_score_col], errors='coerce')
    df = df.dropna(subset=[home_score_col, away_score_col]).copy()
    
    # Create target variables
    df['home_margin'] = df[home_score_col] - df[away_score_col]
    df['home_win'] = (df['home_margin'] > 0).astype(int)
    
    # Drop any rows where targets are missing
    df = df.dropna(subset=['home_margin', 'home_win']).copy()
    
    print(f"\nFinal cleaned dataset: {len(df)} games")
    print(f"Home win rate: {df['home_win'].mean():.1%}")
    print(f"Average home margin: {df['home_margin'].mean():.2f} points")
    
    return df

# Apply the cleaning function
clean_df = prepare_model_data(merged)

# Inspect the results
print("\n" + "="*60)
print("Sample of cleaned data:")
clean_df[['season', 'week', 'home', 'away', 'home_margin', 'home_win']].head(10)

Starting with 16547 games
After filtering to completed games: 16547
After filtering to regular season: 16547
After removing neutral site games: 16109
After removing games with missing scores: 16109

Final cleaned dataset: 16109 games
Home win rate: 62.8%
Average home margin: 7.37 points

Sample of cleaned data:


,season,week,home,away,home_margin,home_win
0,2002,1.0,Virginia,Colorado State,-6.0,0
1,2018,7.0,Auburn,Tennessee,-6.0,0
2,2018,6.0,Ohio State,Indiana,23.0,1
3,2018,6.0,UNLV,New Mexico,-36.0,0
4,2018,6.0,Colorado,Arizona State,7.0,1
5,2018,6.0,Florida Atlantic,Old Dominion,19.0,1
6,2018,6.0,UCF,SMU,28.0,1
7,2018,6.0,Texas A&M,Kentucky,6.0,1
8,2018,6.0,Louisiana Tech,UAB,-21.0,0
9,2018,6.0,Rice,UTSA,-17.0,0


In [32]:
# Summary statistics for targets
print("Home Margin Distribution:")
print(clean_df['home_margin'].describe())
print("\nHome Win Distribution:")
print(clean_df['home_win'].value_counts())

Home Margin Distribution:
count    16109.000000
mean         7.372028
std         22.332248
min        -78.000000
25%         -7.000000
50%          7.000000
75%         23.000000
max         84.000000
Name: home_margin, dtype: float64

Home Win Distribution:
home_win
1    10111
0     5998
Name: count, dtype: int64


## 4. Feature Engineering

Create a feature matrix by:
1. Computing difference features (home stat - away stat) for all numeric paired columns
2. Including venue/travel features if available
3. One-hot encoding categorical variables (conferences)

In [33]:
def build_features(df):
    """
    Engineer features for modeling with intelligent missing data handling.
    
    Creates difference features (home - away) for PRE-GAME stats only.
    EXCLUDES game outcomes to prevent data leakage.
    
    Handles missing data by:
    1. Dropping features that are missing in >50% of rows
    2. Using median imputation for remaining missing values
    
    Parameters:
    -----------
    df : pd.DataFrame
        Cleaned data from prepare_model_data
        
    Returns:
    --------
    X : pd.DataFrame
        Feature matrix
    y_logit : pd.Series
        Binary target (home win)
    y_reg : pd.Series
        Continuous target (home margin)
    """
    feature_df = df.copy()
    
    # List to collect all feature DataFrames
    feature_parts = []
    
    # === CRITICAL: Define features to EXCLUDE (game outcomes) ===
    # These would cause data leakage since they're known only AFTER the game
    EXCLUDE_PATTERNS = [
        'score', 'points', 'margin',  # Final scores
        'q1', 'q2', 'q3', 'q4', 'ot',  # Quarter scores
        'first_downs', 'third_down', 'fourth_down',  # In-game stats
        'pass_comp', 'pass_att', 'pass_yards',  # Game statistics
        'rush_att', 'rush_yards', 'total_yards',  # Game statistics
        'fum', 'int', 'pen_', 'possession',  # Game statistics
        'PostgameWinProbability', 'PostgameElo',  # Post-game metrics
        'excitementIndex',  # Post-game metric
    ]
    
    def should_exclude_feature(col_name):
        """Check if a column should be excluded to prevent data leakage."""
        col_lower = col_name.lower()
        for pattern in EXCLUDE_PATTERNS:
            if pattern.lower() in col_lower:
                return True
        return False
    
    # === 1. Create difference features for home/away stat pairs ===
    # ONLY for pre-game features (e.g., pregame ELO, rankings)
    home_cols = [c for c in feature_df.columns if c.startswith('home') or c.endswith('_home')]
    away_cols = [c for c in feature_df.columns if c.startswith('away') or c.endswith('_away')]
    
    # Find matching pairs
    diff_features = {}
    
    # Try homeXxx / awayXxx pattern (e.g., homePregameElo, awayPregameElo)
    for home_col in home_cols:
        # Skip if this is a game outcome feature
        if should_exclude_feature(home_col):
            continue
            
        if home_col.startswith('home'):
            suffix = home_col[4:]  # Remove 'home' prefix
            away_col = 'away' + suffix
        else:
            # Pattern like rank_home
            prefix = home_col.replace('_home', '')
            away_col = prefix + '_away'
        
        if away_col in feature_df.columns and not should_exclude_feature(away_col):
            # Check if both are numeric
            if pd.api.types.is_numeric_dtype(feature_df[home_col]) and \
               pd.api.types.is_numeric_dtype(feature_df[away_col]):
                # Create difference feature
                feature_name = f'diff_{suffix if home_col.startswith("home") else prefix}'
                diff_features[feature_name] = feature_df[home_col] - feature_df[away_col]
    
    if diff_features:
        diff_df = pd.DataFrame(diff_features)
        
        # Check missingness of each difference feature
        missing_pct = diff_df.isnull().sum() / len(diff_df)
        
        # Keep only features with <50% missing
        good_features = missing_pct[missing_pct < 0.5].index.tolist()
        bad_features = missing_pct[missing_pct >= 0.5].index.tolist()
        
        if bad_features:
            print(f"Dropping {len(bad_features)} difference features with >50% missing data")
        
        diff_df = diff_df[good_features]
        
        if len(diff_df.columns) > 0:
            feature_parts.append(diff_df)
            print(f"Created {len(good_features)} PRE-GAME difference features")
            print(f"  Features: {good_features[:10]}...")  # Show first 10
    
    # === 2. Include venue/travel features if available ===
    venue_features = []
    potential_venue_cols = ['elevation', 'latitude', 'longitude', 'dome', 'grass', 'capacity']
    
    for col in potential_venue_cols:
        if col in feature_df.columns and pd.api.types.is_numeric_dtype(feature_df[col]):
            # Check if feature has <50% missing
            missing_pct = feature_df[col].isnull().sum() / len(feature_df)
            if missing_pct < 0.5:
                venue_features.append(col)
    
    if venue_features:
        venue_df = feature_df[venue_features].copy()
        feature_parts.append(venue_df)
        print(f"Included {len(venue_features)} venue features: {venue_features}")
    
    # === 3. One-hot encode categorical features ===
    categorical_features = []
    
    # Conference features
    if 'homeConference' in feature_df.columns:
        missing_pct = feature_df['homeConference'].isnull().sum() / len(feature_df)
        if missing_pct < 0.5:
            categorical_features.append('homeConference')
    
    if 'awayConference' in feature_df.columns:
        missing_pct = feature_df['awayConference'].isnull().sum() / len(feature_df)
        if missing_pct < 0.5:
            categorical_features.append('awayConference')
    
    # Alternative conference column names
    if 'conf_home' in feature_df.columns:
        missing_pct = feature_df['conf_home'].isnull().sum() / len(feature_df)
        if missing_pct < 0.5:
            categorical_features.append('conf_home')
    
    if 'conf_away' in feature_df.columns:
        missing_pct = feature_df['conf_away'].isnull().sum() / len(feature_df)
        if missing_pct < 0.5:
            categorical_features.append('conf_away')
    
    if categorical_features:
        # Fill missing categorical values with 'Unknown' before encoding
        cat_df = feature_df[categorical_features].copy()
        for col in categorical_features:
            cat_df[col] = cat_df[col].fillna('Unknown')
        
        # One-hot encode, dropping first category to avoid multicollinearity
        encoded_df = pd.get_dummies(
            cat_df, 
            drop_first=True,
            prefix=categorical_features
        )
        feature_parts.append(encoded_df)
        print(f"One-hot encoded {len(categorical_features)} categorical features, " + 
              f"creating {encoded_df.shape[1]} binary columns")
    
    # === 4. Combine all features ===
    if not feature_parts:
        raise ValueError("No features were created! Check the data.")
    
    X = pd.concat(feature_parts, axis=1)
    
    # === 5. Handle missing values with median imputation ===
    rows_before = len(X)
    missing_before = X.isnull().sum().sum()
    
    if missing_before > 0:
        print(f"\nImputing {missing_before:,} missing values using median imputation...")
        
        # Median imputation for numeric features
        for col in X.columns:
            if X[col].isnull().any():
                median_val = X[col].median()
                # If median is NaN (all values missing), use 0
                if pd.isna(median_val):
                    median_val = 0
                X[col] = X[col].fillna(median_val)
    
    # Final check - drop any rows that still have NaN (shouldn't happen, but just in case)
    X = X.dropna()
    rows_after = len(X)
    
    if rows_before > rows_after:
        print(f"Dropped {rows_before - rows_after} rows after imputation (should be minimal)")
    
    # === 6. Extract targets (aligned with X) ===
    y_logit = feature_df.loc[X.index, 'home_win']
    y_reg = feature_df.loc[X.index, 'home_margin']
    
    print(f"\n{'='*60}")
    print(f"Final feature matrix shape: {X.shape}")
    print(f"Number of features: {X.shape[1]}")
    print(f"Data retention: {len(X)/len(df):.1%} of original rows")
    print(f"{'='*60}")
    
    return X, y_logit, y_reg

# Build features
X, y_logit, y_reg = build_features(clean_df)

print("\n" + "="*60)
print(f"Feature matrix shape: {X.shape}")
print(f"Home win rate: {y_logit.mean():.1%}")
print(f"\nHome margin statistics:")
print(y_reg.describe())

Dropping 1 difference features with >50% missing data
Created 2 PRE-GAME difference features
  Features: ['diff_Id', 'diff_PregameElo']...
Included 4 venue features: ['elevation', 'latitude', 'longitude', 'capacity']
One-hot encoded 4 categorical features, creating 77 binary columns

Imputing 3,448 missing values using median imputation...

Final feature matrix shape: (16109, 83)
Number of features: 83
Data retention: 100.0% of original rows

Feature matrix shape: (16109, 83)
Home win rate: 62.8%

Home margin statistics:
count    16109.000000
mean         7.372028
std         22.332248
min        -78.000000
25%         -7.000000
50%          7.000000
75%         23.000000
max         84.000000
Name: home_margin, dtype: float64

Final feature matrix shape: (16109, 83)
Number of features: 83
Data retention: 100.0% of original rows

Feature matrix shape: (16109, 83)
Home win rate: 62.8%

Home margin statistics:
count    16109.000000
mean         7.372028
std         22.332248
min        -

In [34]:
# Preview the features
print("Sample features:")
X.head()

Sample features:


,diff_Id,diff_PregameElo,elevation,latitude,longitude,capacity,homeConference_American Athletic,homeConference_Big 12,homeConference_Big East,homeConference_Big Sky,...,conf_away_big10,conf_away_big12,conf_away_cusa,conf_away_ind,conf_away_mac,conf_away_mwc,conf_away_pac12,conf_away_sec,conf_away_sun-belt,conf_away_wac
0,222,-129.0,170.467743,38.031180,-78.513790,61500.0,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
1,-2631,470.0,201.173798,32.602553,-85.489748,87451.0,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2,110,531.0,216.677032,40.001645,-83.019727,102780.0,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
3,2272,104.0,490.062225,36.085935,-115.017310,36800.0,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
4,29,-23.0,1634.041138,40.009475,-105.266905,50183.0,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False


In [35]:
# DIAGNOSTIC: Verify no data leakage - check features
print("All features created:")
print(X.columns.tolist())
print(f"\nTotal features: {len(X.columns)}")

# Check for any suspicious features
suspicious = []
for col in X.columns:
    col_lower = col.lower()
    if any(word in col_lower for word in ['score', 'points', 'q1', 'q2', 'q3', 'q4', 'postgame', 'yards']):
        suspicious.append(col)

if suspicious:
    print(f"\n⚠️ WARNING: Found {len(suspicious)} potentially leaky features:")
    print(suspicious)
else:
    print("\n✓ No obvious data leakage detected in features")

All features created:
['diff_Id', 'diff_PregameElo', 'elevation', 'latitude', 'longitude', 'capacity', 'homeConference_American Athletic', 'homeConference_Big 12', 'homeConference_Big East', 'homeConference_Big Sky', 'homeConference_Big South', 'homeConference_Big Ten', 'homeConference_CAA', 'homeConference_Conference USA', 'homeConference_FBS Independents', 'homeConference_Ivy', 'homeConference_Mid-American', 'homeConference_Mountain West', 'homeConference_OVC', 'homeConference_Pac-10', 'homeConference_Pac-12', 'homeConference_SEC', 'homeConference_Sun Belt', 'homeConference_Western Athletic', 'awayConference_AWC', 'awayConference_American Athletic', 'awayConference_Atlantic 10', 'awayConference_Atlantic Sun', 'awayConference_Big 12', 'awayConference_Big East', 'awayConference_Big Sky', 'awayConference_Big South', 'awayConference_Big South-OVC', 'awayConference_Big Ten', 'awayConference_CAA', 'awayConference_Conference USA', 'awayConference_FBS Independents', 'awayConference_FCS Indep

## 5. Logistic Regression Model (Home Win Probability)

Train a logistic regression model to predict the probability of a home team victory.
We'll use cross-validation to tune hyperparameters and evaluate performance on a held-out test set.

In [36]:
# Split data for classification task
X_train_logit, X_test_logit, y_train_logit, y_test_logit = train_test_split(
    X, y_logit, 
    test_size=0.2, 
    random_state=RANDOM_STATE,
    stratify=y_logit
)

print(f"Training set: {X_train_logit.shape[0]} games")
print(f"Test set: {X_test_logit.shape[0]} games")
print(f"Training home win rate: {y_train_logit.mean():.1%}")
print(f"Test home win rate: {y_test_logit.mean():.1%}")

Training set: 12887 games
Test set: 3222 games
Training home win rate: 62.8%
Test home win rate: 62.8%


In [37]:
# Build logistic regression pipeline
logit_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

# Define hyperparameter grid
param_grid_logit = {
    'classifier__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'classifier__penalty': ['l2'],
    'classifier__solver': ['lbfgs']
}

# Perform grid search with cross-validation
grid_search_logit = GridSearchCV(
    logit_pipeline,
    param_grid_logit,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

print("Training logistic regression model with GridSearchCV...")
grid_search_logit.fit(X_train_logit, y_train_logit)

print(f"\nBest hyperparameters: {grid_search_logit.best_params_}")
print(f"Best cross-validation ROC AUC: {grid_search_logit.best_score_:.4f}")

# Extract best model
best_logit = grid_search_logit.best_estimator_

Training logistic regression model with GridSearchCV...
Fitting 5 folds for each of 6 candidates, totalling 30 fits

Best hyperparameters: {'classifier__C': 0.01, 'classifier__penalty': 'l2', 'classifier__solver': 'lbfgs'}
Best cross-validation ROC AUC: 0.8262

Best hyperparameters: {'classifier__C': 0.01, 'classifier__penalty': 'l2', 'classifier__solver': 'lbfgs'}
Best cross-validation ROC AUC: 0.8262


In [38]:
# Evaluate on test set
y_pred_logit = best_logit.predict(X_test_logit)
y_pred_proba_logit = best_logit.predict_proba(X_test_logit)[:, 1]

# Calculate metrics
logit_metrics = {
    'accuracy': accuracy_score(y_test_logit, y_pred_logit),
    'roc_auc': roc_auc_score(y_test_logit, y_pred_proba_logit),
    'precision': precision_score(y_test_logit, y_pred_logit),
    'recall': recall_score(y_test_logit, y_pred_logit),
    'f1': f1_score(y_test_logit, y_pred_logit),
    'best_params': grid_search_logit.best_params_
}

print("="*60)
print("LOGISTIC REGRESSION MODEL - TEST SET PERFORMANCE")
print("="*60)
print(f"Accuracy:  {logit_metrics['accuracy']:.4f}")
print(f"ROC AUC:   {logit_metrics['roc_auc']:.4f}")
print(f"Precision: {logit_metrics['precision']:.4f}")
print(f"Recall:    {logit_metrics['recall']:.4f}")
print(f"F1 Score:  {logit_metrics['f1']:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_logit, y_pred_logit))

print("\nClassification Report:")
print(classification_report(y_test_logit, y_pred_logit, target_names=['Away Win', 'Home Win']))

LOGISTIC REGRESSION MODEL - TEST SET PERFORMANCE
Accuracy:  0.7573
ROC AUC:   0.8247
Precision: 0.7828
Recall:    0.8487
F1 Score:  0.8144

Confusion Matrix:
[[ 724  476]
 [ 306 1716]]

Classification Report:
              precision    recall  f1-score   support

    Away Win       0.70      0.60      0.65      1200
    Home Win       0.78      0.85      0.81      2022

    accuracy                           0.76      3222
   macro avg       0.74      0.73      0.73      3222
weighted avg       0.75      0.76      0.75      3222



## 6. Ridge Regression Model (Home Margin)

Train a Ridge regression model to predict the expected point margin for the home team.
We'll use cross-validation to tune the regularization parameter and evaluate performance.

In [39]:
# Split data for regression task
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y_reg,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print(f"Training set: {X_train_reg.shape[0]} games")
print(f"Test set: {X_test_reg.shape[0]} games")
print(f"Training mean home margin: {y_train_reg.mean():.2f} points")
print(f"Test mean home margin: {y_test_reg.mean():.2f} points")

Training set: 12887 games
Test set: 3222 games
Training mean home margin: 7.45 points
Test mean home margin: 7.07 points


In [40]:
# Build Ridge regression pipeline
ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', Ridge(random_state=RANDOM_STATE))
])

# Define hyperparameter grid
param_grid_ridge = {
    'regressor__alpha': [0.001, 0.01, 0.1, 1, 10, 100, 1000]
}

# Perform grid search with cross-validation
grid_search_ridge = GridSearchCV(
    ridge_pipeline,
    param_grid_ridge,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1
)

print("Training Ridge regression model with GridSearchCV...")
grid_search_ridge.fit(X_train_reg, y_train_reg)

print(f"\nBest hyperparameters: {grid_search_ridge.best_params_}")
print(f"Best cross-validation MAE: {-grid_search_ridge.best_score_:.4f}")

# Extract best model
best_reg = grid_search_ridge.best_estimator_

Training Ridge regression model with GridSearchCV...
Fitting 5 folds for each of 7 candidates, totalling 35 fits

Best hyperparameters: {'regressor__alpha': 10}
Best cross-validation MAE: 13.0360

Best hyperparameters: {'regressor__alpha': 10}
Best cross-validation MAE: 13.0360


In [41]:
# Evaluate on test set
y_pred_reg = best_reg.predict(X_test_reg)

# Calculate metrics
mae = mean_absolute_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
r2 = r2_score(y_test_reg, y_pred_reg)

ridge_metrics = {
    'mae': mae,
    'rmse': rmse,
    'r2': r2,
    'best_params': grid_search_ridge.best_params_
}

print("="*60)
print("RIDGE REGRESSION MODEL - TEST SET PERFORMANCE")
print("="*60)
print(f"Mean Absolute Error (MAE):  {mae:.4f} points")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f} points")
print(f"R² Score: {r2:.4f}")

print(f"\nActual mean home margin: {y_test_reg.mean():.2f} points")
print(f"Predicted mean home margin: {y_pred_reg.mean():.2f} points")

RIDGE REGRESSION MODEL - TEST SET PERFORMANCE
Mean Absolute Error (MAE):  13.1557 points
Root Mean Squared Error (RMSE): 16.5803 points
R² Score: 0.4385

Actual mean home margin: 7.07 points
Predicted mean home margin: 7.15 points


## 6b. LASSO Regression Model (Home Margin with Feature Selection)

Train a LASSO regression model to predict the expected point margin for the home team.
LASSO uses L1 regularization which can set some coefficients to exactly zero, providing automatic feature selection.

In [50]:
# Build LASSO regression pipeline (uses same train/test split as Ridge)
lasso_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', Lasso(random_state=RANDOM_STATE, max_iter=10000))
])

# Define hyperparameter grid
param_grid_lasso = {
    'regressor__alpha': [0.001, 0.01, 0.1, 0.5, 1, 5, 10, 50, 100]
}

# Perform grid search with cross-validation
grid_search_lasso = GridSearchCV(
    lasso_pipeline,
    param_grid_lasso,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1
)

print("Training LASSO regression model with GridSearchCV...")
grid_search_lasso.fit(X_train_reg, y_train_reg)

print(f"\nBest hyperparameters: {grid_search_lasso.best_params_}")
print(f"Best cross-validation MAE: {-grid_search_lasso.best_score_:.4f}")

# Extract best model
best_lasso = grid_search_lasso.best_estimator_

Training LASSO regression model with GridSearchCV...
Fitting 5 folds for each of 9 candidates, totalling 45 fits



Best hyperparameters: {'regressor__alpha': 0.001}
Best cross-validation MAE: 13.0358


In [51]:
# Evaluate LASSO on test set
y_pred_lasso = best_lasso.predict(X_test_reg)

# Calculate metrics
mae_lasso = mean_absolute_error(y_test_reg, y_pred_lasso)
rmse_lasso = np.sqrt(mean_squared_error(y_test_reg, y_pred_lasso))
r2_lasso = r2_score(y_test_reg, y_pred_lasso)

lasso_metrics = {
    'mae': mae_lasso,
    'rmse': rmse_lasso,
    'r2': r2_lasso,
    'best_params': grid_search_lasso.best_params_
}

print("="*60)
print("LASSO REGRESSION MODEL - TEST SET PERFORMANCE")
print("="*60)
print(f"Mean Absolute Error (MAE):  {mae_lasso:.4f} points")
print(f"Root Mean Squared Error (RMSE): {rmse_lasso:.4f} points")
print(f"R² Score: {r2_lasso:.4f}")

print(f"\nActual mean home margin: {y_test_reg.mean():.2f} points")
print(f"Predicted mean home margin: {y_pred_lasso.mean():.2f} points")

# Count non-zero coefficients (feature selection)
lasso_coefs = best_lasso.named_steps['regressor'].coef_
n_nonzero = np.sum(lasso_coefs != 0)
print(f"\nFeature Selection: {n_nonzero}/{len(lasso_coefs)} features selected (non-zero coefficients)")

LASSO REGRESSION MODEL - TEST SET PERFORMANCE
Mean Absolute Error (MAE):  13.1560 points
Root Mean Squared Error (RMSE): 16.5809 points
R² Score: 0.4385

Actual mean home margin: 7.07 points
Predicted mean home margin: 7.15 points

Feature Selection: 76/83 features selected (non-zero coefficients)


In [52]:
# Show which features LASSO selected (non-zero coefficients)
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'coefficient': lasso_coefs
}).sort_values('coefficient', key=abs, ascending=False)

print("\nTop 20 Most Important Features (by absolute coefficient):")
print(feature_importance.head(20).to_string(index=False))

print("\nFeatures zeroed out by LASSO:")
zeroed_features = feature_importance[feature_importance['coefficient'] == 0]['feature'].tolist()
print(f"Count: {len(zeroed_features)}")
if len(zeroed_features) > 0 and len(zeroed_features) <= 20:
    print(zeroed_features)
elif len(zeroed_features) > 20:
    print(f"(Too many to display - {len(zeroed_features)} features removed)")


Top 20 Most Important Features (by absolute coefficient):
                         feature  coefficient
                 diff_PregameElo    11.635097
                   conf_away_wac    -5.541789
                 conf_away_pac12    -4.566902
                   conf_away_acc    -4.299003
 awayConference_Western Athletic     3.862472
          awayConference_Big Ten    -3.763665
                   conf_away_ind    -3.597486
              awayConference_SEC    -3.375693
           awayConference_Big 12    -2.985439
             awayConference_MEAC     2.295143
             awayConference_SWAC     2.119902
    awayConference_Mountain West    -1.854709
awayConference_American Athletic    -1.663361
              awayConference_OVC     1.643048
                   conf_away_sec    -1.596946
         awayConference_Big East    -1.584833
                 conf_away_big12    -1.558648
     homeConference_Mid-American    -1.515349
 awayConference_FBS Independents     1.512203
     awayConference_M

## 7. Quantifying Home-Field Advantage

Now we use both models to quantify home-field advantage by simulating a perfectly balanced game where home and away teams are evenly matched (all difference features set to 0).

In [53]:
def quantify_home_advantage_logit(model, X, y):
    """
    Quantify home-field advantage from the logistic regression model.
    
    Returns both a dictionary of metrics and a human-readable summary.
    """
    # Observed home win rate
    observed_win_rate = y.mean()
    
    # Average predicted home win probability across all games
    avg_predicted_prob = model.predict_proba(X)[:, 1].mean()
    
    # Construct a balanced game scenario
    # Set all diff_ features to 0, other features to median
    balanced_game = pd.DataFrame([X.median()], columns=X.columns)
    
    # Set all difference features to 0 (evenly matched teams)
    diff_cols = [col for col in balanced_game.columns if col.startswith('diff_')]
    for col in diff_cols:
        balanced_game[col] = 0
    
    # Predict home win probability for balanced game
    balanced_prob = model.predict_proba(balanced_game)[0, 1]
    
    # Create summary
    results = {
        'observed_win_rate': observed_win_rate,
        'avg_predicted_prob': avg_predicted_prob,
        'balanced_game_prob': balanced_prob,
        'home_advantage_pct_points': (balanced_prob - 0.5) * 100
    }
    
    summary = f"""
HOME-FIELD ADVANTAGE - LOGISTIC MODEL
{'='*60}

Observed Data:
  - Home win rate: {observed_win_rate:.1%}

Model Predictions:
  - Average predicted home win probability: {avg_predicted_prob:.1%}
  
Balanced Game Analysis:
  In a perfectly evenly-matched game (all team stats equal), 
  the model predicts:
  
  - Home team win probability: {balanced_prob:.1%}
  - Away team win probability: {1-balanced_prob:.1%}
  
  Home-field advantage boost: {(balanced_prob - 0.5) * 100:.1f} percentage points
  
INTERPRETATION:
  When two equally-skilled teams play, the home team wins 
  approximately {balanced_prob:.1%} of the time, indicating a 
  significant home-field advantage in college football.
"""
    
    return results, summary

# Calculate and display home-field advantage from logistic model
logit_adv_results, logit_adv_summary = quantify_home_advantage_logit(best_logit, X, y_logit)
print(logit_adv_summary)


HOME-FIELD ADVANTAGE - LOGISTIC MODEL

Observed Data:
  - Home win rate: 62.8%

Model Predictions:
  - Average predicted home win probability: 62.8%
  
Balanced Game Analysis:
  In a perfectly evenly-matched game (all team stats equal), 
  the model predicts:
  
  - Home team win probability: 69.0%
  - Away team win probability: 31.0%
  
  Home-field advantage boost: 19.0 percentage points
  
INTERPRETATION:
  When two equally-skilled teams play, the home team wins 
  approximately 69.0% of the time, indicating a 
  significant home-field advantage in college football.



In [43]:
def quantify_home_advantage_regression(model, X, y):
    """
    Quantify home-field advantage from the Ridge regression model.
    
    Returns both a dictionary of metrics and a human-readable summary.
    """
    # Observed average home margin
    observed_margin = y.mean()
    
    # Average predicted home margin across all games
    avg_predicted_margin = model.predict(X).mean()
    
    # Construct a balanced game scenario
    # Set all diff_ features to 0, other features to median
    balanced_game = pd.DataFrame([X.median()], columns=X.columns)
    
    # Set all difference features to 0 (evenly matched teams)
    diff_cols = [col for col in balanced_game.columns if col.startswith('diff_')]
    for col in diff_cols:
        balanced_game[col] = 0
    
    # Predict home margin for balanced game
    balanced_margin = model.predict(balanced_game)[0]
    
    # Create summary
    results = {
        'observed_margin': observed_margin,
        'avg_predicted_margin': avg_predicted_margin,
        'balanced_game_margin': balanced_margin
    }
    
    summary = f"""
HOME-FIELD ADVANTAGE - RIDGE REGRESSION MODEL
{'='*60}

Observed Data:
  - Average home margin: {observed_margin:+.2f} points

Model Predictions:
  - Average predicted home margin: {avg_predicted_margin:+.2f} points
  
Balanced Game Analysis:
  In a perfectly evenly-matched game (all team stats equal), 
  the model predicts:
  
  - Expected home margin: {balanced_margin:+.2f} points
  
INTERPRETATION:
  When two equally-skilled teams play, the home team is expected 
  to win by approximately {balanced_margin:.1f} points, representing 
  the pure home-field advantage in college football.
"""
    
    return results, summary

# Calculate and display home-field advantage from regression model
reg_adv_results, reg_adv_summary = quantify_home_advantage_regression(best_reg, X, y_reg)
print(reg_adv_summary)


HOME-FIELD ADVANTAGE - RIDGE REGRESSION MODEL

Observed Data:
  - Average home margin: +7.37 points

Model Predictions:
  - Average predicted home margin: +7.39 points
  
Balanced Game Analysis:
  In a perfectly evenly-matched game (all team stats equal), 
  the model predicts:
  
  - Expected home margin: +14.45 points
  
INTERPRETATION:
  When two equally-skilled teams play, the home team is expected 
  to win by approximately 14.4 points, representing 
  the pure home-field advantage in college football.



In [54]:
# Calculate home-field advantage from LASSO model
lasso_adv_results, lasso_adv_summary = quantify_home_advantage_regression(best_lasso, X, y_reg)
print(lasso_adv_summary)


HOME-FIELD ADVANTAGE - RIDGE REGRESSION MODEL

Observed Data:
  - Average home margin: +7.37 points

Model Predictions:
  - Average predicted home margin: +7.39 points
  
Balanced Game Analysis:
  In a perfectly evenly-matched game (all team stats equal), 
  the model predicts:
  
  - Expected home margin: +16.80 points
  
INTERPRETATION:
  When two equally-skilled teams play, the home team is expected 
  to win by approximately 16.8 points, representing 
  the pure home-field advantage in college football.



### Ridge vs. LASSO Comparison

Compare the two regression models side-by-side to determine which is better for quantifying home-field advantage.

In [55]:
# Create comparison table
comparison_df = pd.DataFrame({
    'Metric': ['MAE (points)', 'RMSE (points)', 'R²', 
               'Home Advantage (points)', 'Features Used', 'Best Alpha'],
    'Ridge': [
        f"{ridge_metrics['mae']:.4f}",
        f"{ridge_metrics['rmse']:.4f}",
        f"{ridge_metrics['r2']:.4f}",
        f"{reg_adv_results['balanced_game_margin']:.2f}",
        f"{len(X.columns)}/{len(X.columns)}",
        f"{ridge_metrics['best_params']['regressor__alpha']}"
    ],
    'LASSO': [
        f"{lasso_metrics['mae']:.4f}",
        f"{lasso_metrics['rmse']:.4f}",
        f"{lasso_metrics['r2']:.4f}",
        f"{lasso_adv_results['balanced_game_margin']:.2f}",
        f"{n_nonzero}/{len(X.columns)}",
        f"{lasso_metrics['best_params']['regressor__alpha']}"
    ]
})

print("="*70)
print("RIDGE vs. LASSO REGRESSION COMPARISON")
print("="*70)
print(comparison_df.to_string(index=False))

# Determine winner for each metric
print("\n" + "="*70)
print("INTERPRETATION:")
print("="*70)

# Compare MAE (lower is better)
if ridge_metrics['mae'] < lasso_metrics['mae']:
    mae_winner = "Ridge"
    mae_diff = lasso_metrics['mae'] - ridge_metrics['mae']
else:
    mae_winner = "LASSO"
    mae_diff = ridge_metrics['mae'] - lasso_metrics['mae']

print(f"✓ Prediction Accuracy (MAE): {mae_winner} wins by {mae_diff:.4f} points")

# Compare R² (higher is better)
if ridge_metrics['r2'] > lasso_metrics['r2']:
    r2_winner = "Ridge"
    r2_diff = ridge_metrics['r2'] - lasso_metrics['r2']
else:
    r2_winner = "LASSO"
    r2_diff = lasso_metrics['r2'] - ridge_metrics['r2']

print(f"✓ Variance Explained (R²): {r2_winner} wins by {r2_diff:.4f}")

# Compare interpretability
print(f"✓ Interpretability: LASSO wins (uses only {n_nonzero}/{len(X.columns)} features)")

# Compare home-field advantage estimates
hfa_diff = abs(reg_adv_results['balanced_game_margin'] - lasso_adv_results['balanced_game_margin'])
print(f"✓ Home Advantage Estimate Difference: {hfa_diff:.2f} points")

print("\n" + "="*70)
print("RECOMMENDATION:")
print("="*70)

# Overall recommendation
if mae_diff < 0.1 and n_nonzero < len(X.columns) * 0.8:
    print("→ Use LASSO: Similar prediction accuracy with better interpretability")
    print(f"  ({100 * (len(X.columns) - n_nonzero) / len(X.columns):.0f}% fewer features)")
elif ridge_metrics['mae'] < lasso_metrics['mae'] - 0.5:
    print("→ Use Ridge: Significantly better prediction accuracy")
else:
    print("→ Consider both models or average their predictions")
    print("  Ridge: Better for pure prediction")
    print("  LASSO: Better for identifying key factors")

RIDGE vs. LASSO REGRESSION COMPARISON
                 Metric   Ridge   LASSO
           MAE (points) 13.1557 13.1560
          RMSE (points) 16.5803 16.5809
                     R²  0.4385  0.4385
Home Advantage (points)   14.45   16.80
          Features Used   83/83   76/83
             Best Alpha      10   0.001

INTERPRETATION:
✓ Prediction Accuracy (MAE): Ridge wins by 0.0003 points
✓ Variance Explained (R²): Ridge wins by 0.0000
✓ Interpretability: LASSO wins (uses only 76/83 features)
✓ Home Advantage Estimate Difference: 2.35 points

RECOMMENDATION:
→ Consider both models or average their predictions
  Ridge: Better for pure prediction
  LASSO: Better for identifying key factors


## 8. Save Artifacts

Save all outputs including:
- Model-ready dataset
- Trained models
- Metrics and summaries

In [44]:
# Create necessary directories
os.makedirs('../data/model', exist_ok=True)
os.makedirs('../models', exist_ok=True)
os.makedirs('../reports', exist_ok=True)

print("✓ Directories created/verified")

✓ Directories created/verified


### Save Model-Ready Dataset

In [45]:
# Build model-ready dataset combining identifiers, targets, and features
# Get the aligned subset of clean_df
model_ready_df = clean_df.loc[X.index].copy()

# Select identifier columns
id_cols = []
for col in ['cfbd_game_id', 'sched_game_id', 'season', 'week', 'date', 'home', 'away']:
    if col in model_ready_df.columns:
        id_cols.append(col)

# Combine: identifiers + targets + features
model_df = pd.concat([
    model_ready_df[id_cols],
    pd.DataFrame({'home_margin': y_reg, 'home_win': y_logit}),
    X
], axis=1)

# Save to CSV
output_path = '../data/model/merged_games_model_ready.csv'
model_df.to_csv(output_path, index=False)

print(f"✓ Saved model-ready dataset to {output_path}")
print(f"  Shape: {model_df.shape}")
print(f"  Columns: {len(model_df.columns)} ({len(id_cols)} identifiers + 2 targets + {X.shape[1]} features)")

✓ Saved model-ready dataset to ../data/model/merged_games_model_ready.csv
  Shape: (16109, 92)
  Columns: 92 (7 identifiers + 2 targets + 83 features)


### Save Trained Models

In [ ]:
# Save logistic regression model
logit_model_path = '../models/home_field_logistic.pkl'
joblib.dump(best_logit, logit_model_path)
print(f"✓ Saved logistic regression model to {logit_model_path}")

# Save ridge regression model
ridge_model_path = '../models/home_field_ridge.pkl'
joblib.dump(best_reg, ridge_model_path)
print(f"✓ Saved ridge regression model to {ridge_model_path}")

# Save LASSO regression model
lasso_model_path = '../models/home_field_lasso.pkl'
joblib.dump(best_lasso, lasso_model_path)
print(f"✓ Saved LASSO regression model to {lasso_model_path}")

✓ Saved logistic regression model to ../models/home_field_logistic.pkl
✓ Saved ridge regression model to ../models/home_field_ridge.pkl


### Save Metrics and Summaries

In [ ]:
# Combine all metrics into a single dictionary
all_metrics = {
    'logistic_regression': {
        'test_accuracy': float(logit_metrics['accuracy']),
        'test_roc_auc': float(logit_metrics['roc_auc']),
        'test_precision': float(logit_metrics['precision']),
        'test_recall': float(logit_metrics['recall']),
        'test_f1': float(logit_metrics['f1']),
        'best_params': logit_metrics['best_params'],
        'home_advantage': {
            'observed_win_rate': float(logit_adv_results['observed_win_rate']),
            'avg_predicted_prob': float(logit_adv_results['avg_predicted_prob']),
            'balanced_game_prob': float(logit_adv_results['balanced_game_prob']),
            'home_advantage_pct_points': float(logit_adv_results['home_advantage_pct_points'])
        }
    },
    'ridge_regression': {
        'test_mae': float(ridge_metrics['mae']),
        'test_rmse': float(ridge_metrics['rmse']),
        'test_r2': float(ridge_metrics['r2']),
        'best_params': ridge_metrics['best_params'],
        'features_used': len(X.columns),
        'home_advantage': {
            'observed_margin': float(reg_adv_results['observed_margin']),
            'avg_predicted_margin': float(reg_adv_results['avg_predicted_margin']),
            'balanced_game_margin': float(reg_adv_results['balanced_game_margin'])
        }
    },
    'lasso_regression': {
        'test_mae': float(lasso_metrics['mae']),
        'test_rmse': float(lasso_metrics['rmse']),
        'test_r2': float(lasso_metrics['r2']),
        'best_params': lasso_metrics['best_params'],
        'features_used': int(n_nonzero),
        'features_total': len(X.columns),
        'home_advantage': {
            'observed_margin': float(lasso_adv_results['observed_margin']),
            'avg_predicted_margin': float(lasso_adv_results['avg_predicted_margin']),
            'balanced_game_margin': float(lasso_adv_results['balanced_game_margin'])
        }
    }
}

# Save metrics as JSON
metrics_path = '../reports/model_home_field_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(all_metrics, f, indent=2)

print(f"✓ Saved metrics to {metrics_path}")

# Save logistic model summary as text
logit_summary_path = '../reports/model_home_field_logit_summary.txt'
with open(logit_summary_path, 'w') as f:
    f.write(logit_adv_summary)

print(f"✓ Saved logistic model summary to {logit_summary_path}")

# Save ridge regression model summary as text
ridge_summary_path = '../reports/model_home_field_ridge_summary.txt'
with open(ridge_summary_path, 'w') as f:
    f.write(reg_adv_summary)

print(f"✓ Saved Ridge regression model summary to {ridge_summary_path}")

# Save LASSO regression model summary as text
lasso_summary_path = '../reports/model_home_field_lasso_summary.txt'
with open(lasso_summary_path, 'w') as f:
    f.write(lasso_adv_summary)

print(f"✓ Saved LASSO regression model summary to {lasso_summary_path}")

print("\n" + "="*60)
print("ALL ARTIFACTS SAVED SUCCESSFULLY")
print("="*60)
print("\nSaved outputs:")
print("  - 3 trained models (Logistic, Ridge, LASSO)")
print("  - Model-ready dataset")
print("  - Comprehensive metrics JSON")
print("  - 3 interpretation summaries")

✓ Saved metrics to ../reports/model_home_field_metrics.json
✓ Saved logistic model summary to ../reports/model_home_field_logit_summary.txt
✓ Saved regression model summary to ../reports/model_home_field_reg_summary.txt

ALL ARTIFACTS SAVED SUCCESSFULLY
